# 07: Model Comparison Dashboard

## Learning Goals | 学习目标

1. Build an interactive multi-model comparison dashboard
2. Compare XGBoost vs LEAR vs Persistence on Shandong data
3. Analyze error patterns across time dimensions (hour, day, month)
4. Understand DM/GW statistical tests for model comparison
5. Develop a systematic model evaluation methodology

## Dashboard Overview | 仪表板概览

- **Panel 1**: Load Forecast -- XGBoost vs Persistence overlay + error histogram
- **Panel 2**: Price Forecast -- LEAR vs Persistence overlay + error histogram
- **Panel 3**: Error Analysis -- error heatmap + hourly/weekly/monthly decomposition
- **Panel 4**: Statistical Tests -- DM/GW tests (if epftoolbox available)

All charts support hover interaction -- mouse over to see data details.
所有图表支持悬停交互 -- 鼠标悬停查看数据详情。

In [ ]:
# === Imports | 导入 ===
import warnings
warnings.filterwarnings("ignore")

from ellectric.pipeline.data_loader import create_loader
from ellectric.pipeline.cleaner import clean_data
from ellectric.pipeline.features import FeatureEngineer
from ellectric.pipeline.forecaster import (
    XGBoostForecaster, persistence_forecast
)
from ellectric.pipeline.price_forecaster import LEARForecaster

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

---
## Step 1: Load & Prepare Data | 加载并准备数据

Load Shandong 2024 summer data (Jun-Sep).
Prepare two datasets: one for load forecasting, one for price forecasting.

In [ ]:
print("=== Loading Data ===\n")

# Load Shandong 15-min data
loader = create_loader("shandong")
df_raw = loader.load_data("2024-06-01", "2024-09-30")
df_raw = clean_data(df_raw)

print(f"Shandong data: {len(df_raw):,} rows ({len(df_raw)/96:.0f} days @ 96 pts/day)")
print(f"Time range: {df_raw['timestamp'].min().date()} to {df_raw['timestamp'].max().date()}")
print(f"Columns: {list(df_raw.columns)}")

# --- Prepare load forecasting dataset ---
df_load = df_raw.copy()
fe = FeatureEngineer()
df_load = fe.add_tier1_features(df_load)
df_load = fe.add_tier2_features(df_load)
df_load = fe.add_tier3_features(df_load)
X_cols_load = fe.get_feature_columns("tier3")
print(f"\nLoad features (Tier 3): {len(X_cols_load)} columns")

# --- Prepare price forecasting dataset ---
df_price = df_raw.copy()
df_price["price_da"] = df_price["rt_price"]
df_price["wind_mw"] = df_price["wind_actual_mw"]
df_price["solar_mw"] = df_price["solar_actual_mw"]
df_price = df_price.dropna(subset=["price_da"])

price_forecaster = LEARForecaster(alpha=0.01)
df_price = price_forecaster.add_price_features(df_price, "tier3")
print(f"Price features (Tier 3): {len(price_forecaster.get_feature_columns('tier3'))} columns")

# Display data summary
print(f"\nLoad dataset:  {len(df_load):,} rows")
print(f"Price dataset: {len(df_price):,} rows (null price removed)")
print("\n=== Data Ready ===")

---
## Step 2: Train Models | 训练模型

Train three models on Shandong 15-min data:
1. **Persistence** -- zero training, t-96 shift
2. **XGBoost** -- load forecasting with Tier 3 features
3. **LEAR** -- price forecasting with Tier 3 features

We use TimeSeriesSplit with gap=96 to prevent look-ahead bias.
使用 TimeSeriesSplit，gap=96 防止 look-ahead bias。

In [ ]:
print("=== Training Models ===\n")

# --- 1. Persistence (load) ---
persist_forecast = persistence_forecast(df_load)
persist_load_mae = (persist_forecast - df_load["load_mw"]).abs().mean()
print(f"Persistence Load MAE: {persist_load_mae:.0f} MW")

# --- 2. XGBoost (load) ---
xgb = XGBoostForecaster(n_estimators=100, max_depth=6, learning_rate=0.1)
xgb_result = xgb.train_evaluate(df_load[X_cols_load], df_load["load_mw"], n_splits=5, gap=96)
xgb_mae = xgb_result["metrics"]["mae"]
print(f"XGBoost Load MAE:   {xgb_mae:.0f} MW")
xgb_improve = (persist_load_mae - xgb_mae) / persist_load_mae * 100
print(f"  Improvement vs persistence: {xgb_improve:.1f}%")

# --- 3. LEAR (price) ---
lear_result = price_forecaster.train_evaluate(df_price, tier="tier3", n_splits=5, gap=96)
lear_mae = lear_result["metrics"]["mae"]
print(f"\nLEAR Price MAE:     {lear_mae:.2f} RMB/MWh")

# Persistence for price
persist_price = df_price["price_da"].shift(96)
valid_mask = persist_price.notna()
persist_price_mae = (persist_price[valid_mask] - df_price.loc[valid_mask, "price_da"]).abs().mean()
print(f"Persistence Price MAE: {persist_price_mae:.2f} RMB/MWh")
price_improve = (persist_price_mae - lear_mae) / persist_price_mae * 100
print(f"  Improvement vs persistence: {price_improve:.1f}%")

print("\n=== Training Complete ===")

---
## Panel 1: Load Forecast Comparison | 负荷预测对比

XGBoost vs Persistence overlay + error distribution.
Show the most recent 7 days (7 x 96 = 672 points).

In [ ]:
# --- Panel 1: Load Forecast Dashboard ---
n_show = min(672, len(xgb_result["predictions"]))  # up to 7 days
ts_load = df_load["timestamp"].values[-len(xgb_result["predictions"]):][-n_show:]
actual_load = xgb_result["actuals"][-n_show:]
pred_load = xgb_result["predictions"][-n_show:]
persist_load = persist_forecast.values[-n_show:]
errors_load = actual_load - pred_load

fig_p1 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4], vertical_spacing=0.08,
    subplot_titles=(
        "Load Forecast: XGBoost vs Persistence (Recent 7 Days)",
        "Load Prediction Error Distribution"
    ),
)

# Row 1: Actual vs XGBoost vs Persistence
fig_p1.add_trace(go.Scatter(
    x=ts_load, y=actual_load, mode="lines",
    name="Actual Load", line=dict(color="#1f77b4", width=2)
), row=1, col=1)
fig_p1.add_trace(go.Scatter(
    x=ts_load, y=pred_load, mode="lines",
    name="XGBoost", line=dict(color="#ff7f0e", width=1.5, dash="dash")
), row=1, col=1)
fig_p1.add_trace(go.Scatter(
    x=ts_load, y=persist_load, mode="lines",
    name="Persistence", line=dict(color="#9467bd", width=1, dash="dot")
), row=1, col=1)

# Row 2: Error distribution
fig_p1.add_trace(go.Histogram(
    x=errors_load, nbinsx=40, name="XGBoost Errors",
    marker_color="#2ca02c"
), row=2, col=1)

# Reference lines
median_err = float(np.median(errors_load))
mae_load = float(np.mean(np.abs(errors_load)))
fig_p1.add_vline(x=median_err, line_dash="dash", line_color="red",
    annotation_text=f"Median={median_err:.0f}", row=2, col=1)
fig_p1.add_vline(x=-mae_load, line_dash="dot", line_color="gray", row=2, col=1)
fig_p1.add_vline(x=mae_load, line_dash="dot", line_color="gray",
    annotation_text=f"+/-MAE={mae_load:.0f}", row=2, col=1)

fig_p1.update_layout(
    height=700, hovermode="x unified",
    title="Load Forecast Dashboard -- Shandong 15min"
)
fig_p1.update_xaxes(title_text="Time", row=1, col=1)
fig_p1.update_xaxes(title_text="Error (MW)", row=2, col=1)
fig_p1.update_yaxes(title_text="Load (MW)", row=1, col=1)
fig_p1.update_yaxes(title_text="Count", row=2, col=1)

fig_p1.show()

print(f"XGBoost Load MAE: {xgb_mae:.0f} MW")
print(f"Persistence Load MAE: {persist_load_mae:.0f} MW")
print(f"Improvement: {xgb_improve:.1f}%")

---
## Panel 2: Price Forecast Comparison | 电价预测对比

LEAR vs Persistence overlay + error distribution.

In [ ]:
# --- Panel 2: Price Forecast Dashboard ---
n_show_p = min(672, len(lear_result["predictions"]))  # up to 7 days
ts_price = df_price["timestamp"].values[-len(lear_result["predictions"]):][-n_show_p:]
actual_price = lear_result["actuals"][-n_show_p:]
pred_price = lear_result["predictions"][-n_show_p:]
errors_price = actual_price - pred_price

fig_p2 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.6, 0.4], vertical_spacing=0.08,
    subplot_titles=(
        "Price Forecast: LEAR vs Persistence (Recent 7 Days)",
        "Price Prediction Error Distribution"
    ),
)

# Row 1
fig_p2.add_trace(go.Scatter(
    x=ts_price, y=actual_price, mode="lines",
    name="Actual Price", line=dict(color="#1f77b4", width=2)
), row=1, col=1)
fig_p2.add_trace(go.Scatter(
    x=ts_price, y=pred_price, mode="lines",
    name="LEAR (Lasso)", line=dict(color="#ff7f0e", width=1.5, dash="dash")
), row=1, col=1)

# Row 2
fig_p2.add_trace(go.Histogram(
    x=errors_price, nbinsx=40, name="LEAR Errors",
    marker_color="#d62728"
), row=2, col=1)

median_err_p = float(np.median(errors_price))
fig_p2.add_vline(x=median_err_p, line_dash="dash", line_color="red",
    annotation_text=f"Median={median_err_p:.1f}", row=2, col=1)
fig_p2.add_vline(x=-lear_mae, line_dash="dot", line_color="gray", row=2, col=1)
fig_p2.add_vline(x=lear_mae, line_dash="dot", line_color="gray",
    annotation_text=f"+/-MAE={lear_mae:.1f}", row=2, col=1)

fig_p2.update_layout(
    height=700, hovermode="x unified",
    title="Price Forecast Dashboard -- Shandong 15min"
)
fig_p2.update_xaxes(title_text="Time", row=1, col=1)
fig_p2.update_xaxes(title_text="Error (RMB/MWh)", row=2, col=1)
fig_p2.update_yaxes(title_text="Price (RMB/MWh)", row=1, col=1)
fig_p2.update_yaxes(title_text="Count", row=2, col=1)

fig_p2.show()

print(f"LEAR Price MAE:     {lear_mae:.2f} RMB/MWh")
print(f"Persistence Price MAE: {persist_price_mae:.2f} RMB/MWh")
print(f"Improvement: {price_improve:.1f}%")

---
## Panel 3: Error Analysis by Time | 按时间维度误差分析

Decompose prediction errors by hour, weekday, and month.
This reveals whether models have systematic biases at certain times.

按小时/星期/月份分解误差，揭示模型是否有系统性偏差。

In [ ]:
# --- Panel 3: Multi-Dimensional Error Analysis ---
n_pred_len = len(xgb_result["predictions"])

# Build error DataFrame
error_df = pd.DataFrame({
    "timestamp": df_load["timestamp"].values[-n_pred_len:],
    "error": xgb_result["actuals"] - xgb_result["predictions"],
})
error_df["hour"] = pd.DatetimeIndex(error_df["timestamp"]).hour
error_df["date"] = pd.DatetimeIndex(error_df["timestamp"]).date
error_df["day_name"] = pd.DatetimeIndex(error_df["timestamp"]).day_name()
error_df["month"] = pd.DatetimeIndex(error_df["timestamp"]).month

# --- Hourly MAE ---
hourly_mae = error_df.groupby("hour")["error"].apply(lambda x: float(np.abs(x).mean()))

# --- Weekday MAE ---
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
weekday_cn = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_mae = error_df.groupby("day_name")["error"].apply(lambda x: float(np.abs(x).mean()))
weekday_vals = [weekday_mae.get(d, 0) for d in weekday_order]

# --- Monthly MAE ---
month_mae = error_df.groupby("month")["error"].apply(lambda x: float(np.abs(x).mean()))

fig_p3 = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "XGBoost MAE by Hour",
        "XGBoost MAE by Weekday",
        "XGBoost MAE by Month"
    ),
    column_widths=[0.4, 0.3, 0.3],
)

# Hourly
fig_p3.add_trace(go.Bar(
    x=list(range(24)), y=hourly_mae.values,
    marker_color="#1f77b4", name="Hourly MAE",
    text=[f"{v:.0f}" for v in hourly_mae.values], textposition="outside",
), row=1, col=1)

# Weekday
fig_p3.add_trace(go.Bar(
    x=weekday_cn, y=weekday_vals,
    marker_color=["#1f77b4"] * 5 + ["#ff7f0e"] * 2, name="Weekday MAE",
    text=[f"{v:.0f}" for v in weekday_vals], textposition="outside",
), row=1, col=2)

# Monthly
months_present = sorted(month_mae.index.tolist())
fig_p3.add_trace(go.Bar(
    x=[f"M{m}" for m in months_present],
    y=[month_mae[m] for m in months_present],
    marker_color="#2ca02c", name="Monthly MAE",
    text=[f"{month_mae[m]:.0f}" for m in months_present], textposition="outside",
), row=1, col=3)

fig_p3.update_layout(
    height=400, showlegend=False,
    title="Error Analysis by Time Dimension -- XGBoost Load Forecast"
)
fig_p3.update_yaxes(title_text="MAE (MW)", row=1, col=1)
fig_p3.update_yaxes(title_text="MAE (MW)", row=1, col=2)
fig_p3.update_yaxes(title_text="MAE (MW)", row=1, col=3)

fig_p3.show()

# Print insights
print("Error Analysis Summary:")
print(f"  Best hour:  {hourly_mae.idxmin()}:00 (MAE={hourly_mae.min():.0f} MW)")
print(f"  Worst hour: {hourly_mae.idxmax()}:00 (MAE={hourly_mae.max():.0f} MW)")
if len(weekday_vals) >= 7:
    best_day = weekday_cn[int(np.argmin(weekday_vals))]
    worst_day = weekday_cn[int(np.argmax(weekday_vals))]
    print(f"  Best day:   {best_day} (MAE={min(weekday_vals):.0f} MW)")
    print(f"  Worst day:  {worst_day} (MAE={max(weekday_vals):.0f} MW)")

---
## Panel 4: Model Comparison Bar + Statistical Tests | 模型对比 + 统计检验

Compare all models side-by-side. Run DM/GW tests if epftoolbox is installed.

### Diebold-Mariano (DM) Test
H0: Two forecasts have equal predictive accuracy.
p < 0.05 -> reject H0 -> one model is significantly better.

### Giacomini-White (GW) Test
H0: Baseline model is not inferior to challenger.
p < 0.05 -> reject H0 -> challenger is significantly better.

对比所有模型。如果安装了 epftoolbox 则运行 DM/GW 检验。

In [ ]:
# --- Panel 4: Model Comparison Bar Chart ---
model_names = ["Persistence\n(Load)", "XGBoost\n(Load)", "Persistence\n(Price)", "LEAR\n(Price)"]
model_values = [persist_load_mae, xgb_mae, persist_price_mae, lear_mae]
model_units = ["MW", "MW", "RMB/MWh", "RMB/MWh"]
model_colors = ["#9467bd", "#ff7f0e", "#9467bd", "#d62728"]

fig_p4 = make_subplots(
    rows=1, cols=2,
    column_widths=[0.5, 0.5],
    subplot_titles=("Model MAE Comparison", "Improvement over Persistence (%)"),
)

# Left: MAE comparison
fig_p4.add_trace(go.Bar(
    x=model_names, y=model_values,
    marker_color=model_colors,
    text=[f"{v:.1f}" for v in model_values],
    textposition="outside",
    name="MAE",
), row=1, col=1)

# Right: Improvement %
improvements = [0, xgb_improve, 0, price_improve]
improve_labels = ["Persistence\n(Load)", "XGBoost\n(Load)", "Persistence\n(Price)", "LEAR\n(Price)"]
improve_colors = ["#9467bd", "#2ca02c" if xgb_improve > 0 else "#d62728",
                  "#9467bd", "#2ca02c" if price_improve > 0 else "#d62728"]

fig_p4.add_trace(go.Bar(
    x=improve_labels, y=improvements,
    marker_color=improve_colors,
    text=[f"{v:.1f}%" for v in improvements],
    textposition="outside",
    name="Improvement %",
), row=1, col=2)

fig_p4.update_layout(
    height=450, showlegend=False,
    title="Model Performance Summary -- Shandong 15min"
)
fig_p4.update_yaxes(title_text="MAE", row=1, col=1)
fig_p4.update_yaxes(title_text="Improvement (%)", row=1, col=2)

fig_p4.show()

# Ranking
print("Model Ranking (Load):")
print(f"  1. XGBoost:     {xgb_mae:.0f} MW (baseline)")
print(f"  2. Persistence: {persist_load_mae:.0f} MW [{xgb_improve:+.1f}%]")
print(f"\nModel Ranking (Price):")
print(f"  1. LEAR:        {lear_mae:.2f} RMB/MWh (baseline)")
print(f"  2. Persistence: {persist_price_mae:.2f} RMB/MWh [{price_improve:+.1f}%]")

In [ ]:
# --- DM/GW Statistical Tests (optional) | DM/GW 统计检验 (可选) ---
print("=== DM/GW Statistical Tests ===\n")

try:
    from ellectric.pipeline.statistical_tests import run_statistical_tests

    # Prepare LEAR error series
    lear_errors_all = lear_result["actuals"] - lear_result["predictions"]

    # Prepare persistence error series
    persist_price_vals = df_price["price_da"].shift(96)
    valid_mask = persist_price_vals.notna()
    persist_errors_all = (persist_price_vals[valid_mask] - df_price.loc[valid_mask, "price_da"]).values

    # Run DM/GW tests: LEAR vs Persistence on Shandong data
    # Use mock benchmarks (since epftoolbox may not be installed)
    test_results = run_statistical_tests(
        errors_chinese=lear_errors_all,
        errors_benchmarks={
            "EPEX-BE": persist_errors_all[:len(lear_errors_all)],
            "NordPool": persist_errors_all[:len(lear_errors_all)],
            "PJM": persist_errors_all[:len(lear_errors_all)],
        },
        h=96,
        crit="MAE",
    )

    print(test_results["summary"])

    # Note about mock data
    if any("MOCK" in str(r.get("skip_reason", "")) for r in test_results["dm_results"]):
        print("\nNote: DM/GW results use mock data because epftoolbox is not installed.")
        print("Install with: pip install git+https://github.com/jeslago/epftoolbox.git")
        print("注意: epftoolbox 未安装，DM/GW 结果使用模拟数据。")

except ImportError:
    print("statistical_tests module not found -- skipping DM/GW tests")
    print("statistical_tests 模块未找到 -- 跳过 DM/GW 检验")
except Exception as e:
    print(f"DM/GW tests failed: {e}")
    print(f"DM/GW 检验失败: {e}")

---
## Key Takeaways | 学习笔记

- **XGBoost vs Persistence**: XGBoost should beat persistence on load forecasting
  if features capture diurnal/weekly patterns.
- **LEAR vs Persistence**: Price forecasting is harder; LEAR uses L1 to select
  predictive features automatically.
- **Error decomposition** reveals systematic biases -- if errors peak at certain
  hours, the model is missing something about those periods.
- **DM/GW tests** add statistical rigor: is the improvement significant or just
  noise? p < 0.05 means the difference is unlikely to be random.
- **15-min data** provides much richer signal than hourly data.

## Cross-Panel Reflection | 跨面板反思

1. Panel 1 -> Panel 3: Do the largest errors in the histogram correspond to
   specific hours in the hourly decomposition?
2. Panel 1 -> Panel 2: Is load forecasting easier or harder than price forecasting?
   Why?
3. Panel 3 -> Panel 4: If a model shows clear time-of-day bias, is a statistical
   test enough to call it "good"?
4. Overall: What would you change first if you had to improve the weakest model?

Next: Notebook 08 -> ASSUME Simulation Results Analysis.